In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer # т.н. преобразователь колонок
from sklearn.linear_model import SGDRegressor, L
from sklearn.metrics import root_mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
import pickle
import mlflow
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from mlflow.models import infer_signature
from sklearn.model_selection import GridSearchCV

In [2]:
def preprocessing_data_frame(frame):
    df = frame.copy()
    cat_columns = ['Make', 'Model', 'Style', 'Fuel_type', 'Transmission']
    num_columns = ['Year', 'Distance', 'Engine_capacity(cm3)', 'Price(euro)']
    
    question_dist = df[(df.Year <2021) & (df.Distance < 1100)]
    df = df.drop(question_dist.index)
    # Анализ и очистка данных
    # анализ гистограмм
    question_dist = df[(df.Distance > 1e6)]
    df = df.drop(question_dist.index)
    
    # здравый смысл
    question_engine = df[df["Engine_capacity(cm3)"] < 200]
    df = df.drop(question_engine.index)
    
    # здравый смысл
    question_engine = df[df["Engine_capacity(cm3)"] > 5000]
    df = df.drop(question_engine.index)
    
    # здравый смысл
    question_price = df[(df["Price(euro)"] < 101)]
    df = df.drop(question_price.index)
    
    # анализ гистограмм
    question_price = df[df["Price(euro)"] > 1e5]
    df = df.drop(question_price.index)
    
    #анализ гистограмм
    question_year = df[df.Year < 1971]
    df = df.drop(question_year.index)
    
    df = df.reset_index(drop=True)  # обновим индексы в датафрейме DF. если бы мы прописали drop = False, то была бы еще одна колонка - старые индексы
    # Разделение данных на признаки и целевую переменную
    
    
    # Предварительная обработка категориальных данных
    # Порядковое кодирование. Обучение, трансформация и упаковка в df
    
    ordinal = OrdinalEncoder()
    ordinal.fit(df[cat_columns]);
    Ordinal_encoded = ordinal.transform(df[cat_columns])
    df_ordinal = pd.DataFrame(Ordinal_encoded, columns=cat_columns)
    df[cat_columns] = df_ordinal[cat_columns]
    return df

def scale_frame(frame):
    df = frame.copy()
    X,y = df.drop(columns = ['Price(euro)']), df['Price(euro)']
    scaler = StandardScaler()
    power_trans = PowerTransformer()
    X_scale = scaler.fit_transform(X.values)
    Y_scale = power_trans.fit_transform(y.values.reshape(-1,1))
    return X_scale, Y_scale, power_trans

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/dayekb/Basic_ML_Alg/main/cars_moldova_no_dup.csv', delimiter = ',')
df.head()

,Make,Model,Year,Style,Distance,Engine_capacity(cm3),Fuel_type,Transmission,Price(euro)
0,Toyota,Prius,2011,Hatchback,195000.0,1800.0,Hybrid,Automatic,7750.0
1,Renault,Grand Scenic,2014,Universal,135000.0,1500.0,Diesel,Manual,8550.0
2,Volkswagen,Golf,1998,Hatchback,1.0,1400.0,Petrol,Manual,2200.0
3,Renault,Laguna,2012,Universal,110000.0,1500.0,Diesel,Manual,6550.0
4,Opel,Astra,2006,Universal,200000.0,1600.0,Metan/Propan,Manual,4100.0


In [4]:
df_proc = preprocessing_data_frame(df)
X,Y, power_trans = scale_frame(df_proc)
# разбиваем на тестовую и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(X, Y,
                                                  test_size=0.3,
                                                  random_state=42)

In [5]:
def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2

In [62]:
params = {'alpha': [0.0001, 0.001, 0.01, 0.05, 0.1 ],
      'l1_ratio': [0.001, 0.05, 0.01, 0.2]
 }
with mlflow.start_run():
    lr = SGDRegressor(random_state=42)
    clf = GridSearchCV(lr, params, cv = 5)
    clf.fit(X_train, y_train.reshape(-1))
    best = clf.best_estimator_
    y_pred = best.predict(X_val)
    y_price_pred = power_trans.inverse_transform(y_pred.reshape(-1,1))
    (rmse, mae, r2)  = eval_metrics(power_trans.inverse_transform(y_val), y_price_pred)
    alpha = best.alpha
    l1_ratio = best.l1_ratio
    mlflow.log_param("alpha", alpha)
    mlflow.log_param("l1_ratio", l1_ratio)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("mae", mae)
    
    predictions = best.predict(X_train)
    signature = infer_signature(X_train, predictions)
    mlflow.sklearn.log_model(best, "model", signature=signature)

In [74]:
power_trans.inverse_transform(best.predict([X_val[122]]).reshape(-1,1))

array([[1443.7366942]])

In [75]:
!ls ./mlruns/0/c03b5c67e69b4d4f854b6f125254dd27/artifacts/model/

conda.yaml  MLmodel  model.pkl	python_env.yaml  requirements.txt


In [101]:
## История запуска моделей

In [99]:
frame = dfruns.sort_values("metrics.r2", ascending=False)
frame

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.r2,metrics.rmse,metrics.mae,params.alpha,params.l1_ratio,tags.mlflow.log-model.history,tags.mlflow.user,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.source.name
0,d9f5c7d863d24d1591830bb7b15fd01b,0,FINISHED,file:///media/kirilman/Z3/URFU/MLOPS/%D0%9B%D0...,2026-03-25 06:30:11.841000+00:00,2026-03-25 06:30:15.620000+00:00,0.623362,5834.294652,2949.176298,0.0001,0.001,"[{""run_id"": ""d9f5c7d863d24d1591830bb7b15fd01b""...",kirilman,LOCAL,thoughtful-sow-230,/home/kirilman/.local/lib/python3.9/site-packa...
1,d1a54c46d9db4f64b54c41686c2a8298,0,FINISHED,file:///media/kirilman/Z3/URFU/MLOPS/%D0%9B%D0...,2026-03-25 06:29:55.138000+00:00,2026-03-25 06:29:58.469000+00:00,0.623362,5834.294652,2949.176298,0.0001,0.001,"[{""run_id"": ""d1a54c46d9db4f64b54c41686c2a8298""...",kirilman,LOCAL,clean-shrew-212,/home/kirilman/.local/lib/python3.9/site-packa...
2,24b2593183e54e06b2774a0b5adba7e6,0,FINISHED,file:///media/kirilman/Z3/URFU/MLOPS/%D0%9B%D0...,2026-03-25 06:25:13.224000+00:00,2026-03-25 06:25:16.051000+00:00,0.623362,5834.294652,2949.176298,0.0001,0.001,"[{""run_id"": ""24b2593183e54e06b2774a0b5adba7e6""...",kirilman,LOCAL,skittish-trout-79,/home/kirilman/.local/lib/python3.9/site-packa...
3,79df98b8ab0c408c89f1943e0b80fab6,0,FINISHED,file:///media/kirilman/Z3/URFU/MLOPS/%D0%9B%D0...,2026-03-25 06:24:02.647000+00:00,2026-03-25 06:24:05.642000+00:00,0.623362,5834.294652,2949.176298,0.0001,0.001,"[{""run_id"": ""79df98b8ab0c408c89f1943e0b80fab6""...",kirilman,LOCAL,grandiose-gnat-455,/home/kirilman/.local/lib/python3.9/site-packa...
6,c03b5c67e69b4d4f854b6f125254dd27,0,FINISHED,file:///media/kirilman/Z3/URFU/MLOPS/%D0%9B%D0...,2026-03-25 06:09:22.264000+00:00,2026-03-25 06:09:25.957000+00:00,0.623362,5834.294652,2949.176298,0.0001,0.001,"[{""run_id"": ""c03b5c67e69b4d4f854b6f125254dd27""...",kirilman,LOCAL,rumbling-cub-948,/home/kirilman/.local/lib/python3.9/site-packa...
7,6bf215d7c8db4444bae326ac4a533fa8,0,FINISHED,file:///media/kirilman/Z3/URFU/MLOPS/%D0%9B%D0...,2026-03-25 05:57:53.201000+00:00,2026-03-25 05:57:57.200000+00:00,0.623362,5834.294652,2949.176298,0.0001,0.001,"[{""run_id"": ""6bf215d7c8db4444bae326ac4a533fa8""...",kirilman,LOCAL,masked-goat-727,/home/kirilman/.local/lib/python3.9/site-packa...
4,7f1b77fc5edd4bc08dcbe7eafb7bab87,0,FAILED,file:///media/kirilman/Z3/URFU/MLOPS/%D0%9B%D0...,2026-03-25 06:23:43.113000+00:00,2026-03-25 06:23:43.121000+00:00,NaN,NaN,NaN,None,None,None,kirilman,LOCAL,grandiose-asp-595,/home/kirilman/.local/lib/python3.9/site-packa...
5,20200c665caf4a8eb7d1bfa8b6ca76dd,0,FAILED,file:///media/kirilman/Z3/URFU/MLOPS/%D0%9B%D0...,2026-03-25 06:23:37.807000+00:00,2026-03-25 06:23:37.812000+00:00,NaN,NaN,NaN,None,None,None,kirilman,LOCAL,vaunted-finch-871,/home/kirilman/.local/lib/python3.9/site-packa...


In [96]:
dfruns = mlflow.search_runs()
model_uri = dfruns.sort_values("metrics.r2", ascending=False).iloc[0]['artifact_uri']
path2model = dfruns.sort_values("metrics.r2", ascending=False).iloc[0]['artifact_uri'].replace("file://","") + '/model' #путь до эксперимента с лучшей моделью
print(path2model)

/media/kirilman/Z3/URFU/MLOPS/%D0%9B%D0%B0%D0%B1%D1%8B/MLOPS/mlflow/mlruns/0/d9f5c7d863d24d1591830bb7b15fd01b/artifacts/model


In [95]:
loaded_model = mlflow.sklearn.load_model(path2model)

test_input = np.array([[0.08265284, -0.18711058, 0.51492644, -1.41445941, 
                        0.28915819, 0.54708212, -1.01395689, -1.09462628]])
prediction = loaded_model.predict(test_input)
prediction

array([0.71926926])

### Запускает модель в сервис локально 

In [68]:
!mlflow models serve -m /media/kirilman/Z3/URFU/MLOPS/Лабы/MLOPS/mlflow/mlruns/0/d9f5c7d863d24d1591830bb7b15fd01b/artifacts/model/ -p 5001 --env-manager local

2026/03/25 11:31:09 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2026/03/25 11:31:09 INFO mlflow.pyfunc.backend: === Running command 'exec gunicorn --timeout=60 -b 127.0.0.1:5001 -w 1 ${GUNICORN_CMD_ARGS} -- mlflow.pyfunc.scoring_server.wsgi:app'
[2026-03-25 11:31:09 +0500] [281807] [INFO] Starting gunicorn 21.2.0
[2026-03-25 11:31:09 +0500] [281807] [INFO] Listening at: http://127.0.0.1:5001 (281807)
[2026-03-25 11:31:09 +0500] [281807] [INFO] Using worker: sync
[2026-03-25 11:31:09 +0500] [281808] [INFO] Booting worker with pid: 281808
^C
[2026-03-25 11:31:28 +0500] [281807] [INFO] Handling signal: int
[2026-03-25 11:31:28 +0500] [281808] [INFO] Worker exiting (pid: 281808)


### Тестовый запрос к модели через curl

In [19]:
!curl http://127.0.0.1:5001/invocations -H"Content-Type:application/json"  --data '{"inputs": [[ 0.08265284, -0.18711058,  0.51492644, -1.41445941,  0.28915819, 0.54708212, -1.01395689, -1.09462628 ]]}'

mlflow.ipynb  mlruns
